
# Choose your corpus
Pick a collection of text: Wikipedia articles on AI/ML topics, Pakistani news articles, research paper abstracts, or any domain you care about. Aim for 200–1000 documents. Clean the text.

In [2]:
import pandas as pd
df = pd.read_csv("/kaggle/input/datasets/zaidrahim/pakistani-news/Pakistan_News.csv")
columns_names=['Date',"Source","Country (Profile)"]
df=df.drop(columns=columns_names,errors='ignore')


In [3]:
df[:30]

,Details
0,CPJ urges India to stop harassing employees of...
1,Feature: Chronic illnesses put jailed PFI foun...
2,Kashmiris to observe Jammu Martyrs Day on Nove...
3,Modi regime using brutal tactics to harass jou...
4,Ladakh protests in support of rights #Pakistan...
5,Saghar condemns India’s nefarious designs in I...
6,There are dozens of companies containing word ...
7,Article: Why is the Anniversary of the 1984 Ma...
8,"Article: Morbid Morbi, immoral media and a dem..."
9,Article: ‘Disproportionate’: Legal experts cri...


**Data pre Processing**

In [4]:
documents = df["Details"].fillna("").astype(str).tolist()

In [5]:
print(type(documents))

<class 'list'>


In [6]:
documents[0]

'CPJ urges India to stop harassing employees of The Wire #Pakistan #Kashmir #IndependentKashmir kmsnews.org/kms/2022/11/02…'

In [7]:
len(df)

1852

In [8]:
documents[:30]

['CPJ urges India to stop harassing employees of The Wire #Pakistan #Kashmir #IndependentKashmir kmsnews.org/kms/2022/11/02…',
 'Feature: Chronic illnesses put jailed PFI founder E Abubacker’s “life at risk” #Pakistan #Kashmir #IndependentKashmir kmsnews.org/kms/2022/11/02…',
 'Kashmiris to observe Jammu Martyrs Day on November 6 #Pakistan #Kashmir #IndependentKashmir kmsnews.org/kms/2022/11/02…',
 'Modi regime using brutal tactics to harass journalists in IIOJK, India #Pakistan #Kashmir #IndependentKashmir kmsnews.org/kms/2022/11/02…',
 'Ladakh protests in support of rights #Pakistan #Kashmir #IndependentKashmir kmsnews.org/kms/2022/11/02…',
 'Saghar condemns India’s nefarious designs in IIOJK #Pakistan #Kashmir #IndependentKashmir kmsnews.org/kms/2022/11/02…',
 'There are dozens of companies containing word Jinnah (including Jinnah newspaper). What will be the next move? Ban companies having "Pakistan" in name? twitter.com/misterzedpk/st…',
 'Article: Why is the Anniversary of the 19

In [9]:
import re
import html

def clean_text(text):
    if not isinstance(text, str):
        return ""

    text = html.unescape(text)

    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Remove URL fragments like /kms/2022/11/02
    text = re.sub(r'/\S+', '', text)

    # Remove @mentions
    text = re.sub(r'@\w+', '', text)

    # Convert hashtags to words
    text = re.sub(r'#(\w+)', r'\1', text)

    # Remove repeated whitespace
    text = re.sub(r'\s+', ' ', text)

    return text.strip()
documents = [clean_text(doc) for doc in documents]

In [10]:
documents[:10]

['CPJ urges India to stop harassing employees of The Wire Pakistan Kashmir IndependentKashmir kmsnews.org',
 'Feature: Chronic illnesses put jailed PFI founder E Abubacker’s “life at risk” Pakistan Kashmir IndependentKashmir kmsnews.org',
 'Kashmiris to observe Jammu Martyrs Day on November 6 Pakistan Kashmir IndependentKashmir kmsnews.org',
 'Modi regime using brutal tactics to harass journalists in IIOJK, India Pakistan Kashmir IndependentKashmir kmsnews.org',
 'Ladakh protests in support of rights Pakistan Kashmir IndependentKashmir kmsnews.org',
 'Saghar condemns India’s nefarious designs in IIOJK Pakistan Kashmir IndependentKashmir kmsnews.org',
 'There are dozens of companies containing word Jinnah (including Jinnah newspaper). What will be the next move? Ban companies having "Pakistan" in name? twitter.com',
 'Article: Why is the Anniversary of the 1984 Massacre of Sikhs Not a Time of Remembrance For Us All? Pakistan Kashmir IndependentKashmir kmsnews.org',
 'Article: Morbid Mor

# Model Part
**Embed & index all documents**
Generate embeddings for every document using sentence-transformers. Store in a NumPy matrix. Save to disk so you don't re-embed every run. This indexing step mirrors what vector databases do.

In [11]:
from sentence_transformers import SentenceTransformer

In [12]:
model=SentenceTransformer("sentence-transformers/all-mpnet-base-v2",device="cuda")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
embeddings=model.encode_document(documents)

In [14]:
embeddings[:5]

array([[ 0.00352344,  0.03393032,  0.00327199, ...,  0.00922078,
        -0.06121768,  0.00180131],
       [ 0.00501876,  0.11759647, -0.00730387, ...,  0.00022264,
        -0.05076964, -0.03266895],
       [ 0.01203159, -0.00885716, -0.00160439, ...,  0.01895912,
        -0.01168302,  0.04150926],
       [ 0.07134759,  0.02458234, -0.01370628, ...,  0.03279904,
        -0.04270622, -0.00851341],
       [-0.02163709, -0.00329872, -0.00713771, ...,  0.0344375 ,
        -0.00462   ,  0.03553348]], dtype=float32)

In [15]:
import numpy as np

In [17]:
array=np.array(embeddings)

In [18]:
array.shape

(1852, 768)

In [21]:
np.save("/kaggle/working/Doc_Embeddings.npy",embeddings)

# Build Streamlit search UI
Text input for query → embed query → compute similarity vs all docs → show top 5 results with similarity scores. Add a bar chart of scores. This is a complete working semantic search product

In [28]:
def cosine_similarity(a, b):
    # np.dot(a, b) is the dot product
    # np.linalg.norm(a) calculates the square root of the sum of squares
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [31]:
def sementic_search(query,top_k=5):
    query=model.encode_query(query)
    similarity = [cosine_similarity(query, doc_emb) for doc_emb in embeddings]
    top=sorted(enumerate(similarity), key=lambda x:x[1],reverse=True)
    print(f"\nQuery: '{query}'")
    print("-" * 50)
    for top, (idx, similarity) in enumerate(top[:top_k], 1):
        print(f"{top}. [{similarity:.3f}] {documents[idx]}")

In [32]:
sementic_search("")


Query: '[ 4.83809225e-02  2.38646585e-02 -9.98979621e-03 -1.93919451e-03
 -4.49877940e-02  8.52300599e-03 -1.52452281e-02  2.07849815e-02
  4.33633476e-02  3.35809626e-02 -1.16118332e-02  1.84730384e-02
 -1.39208818e-02 -7.07571507e-02 -1.44674312e-02  3.54163870e-02
 -1.44240772e-03  1.36370482e-02  1.81988105e-02  4.32059802e-02
 -3.55121419e-02  3.90328048e-03 -2.66919639e-02  2.32805479e-02
 -1.08710546e-02 -9.54884198e-03  7.09261224e-02 -3.64604183e-02
 -6.17835149e-02 -4.61666612e-03 -5.44872833e-03  1.59557294e-02
  5.84696373e-03 -4.70886454e-02  1.10140331e-06 -3.07395253e-02
  2.91185901e-02  1.04262559e-02 -7.15110160e-04 -4.05283528e-04
 -7.61158392e-02  2.29514334e-02 -1.18321204e-03  1.74851026e-02
 -6.12232573e-02  3.72162685e-02  3.48872459e-03  6.90163206e-03
  5.52419433e-03  1.02820456e-01  7.29597686e-03  1.29297376e-02
  5.01988782e-03 -2.94360127e-02  4.13312539e-02 -2.40776781e-02
 -8.53808317e-03  3.67881916e-02  1.53684542e-02 -5.64116566e-03
  2.23129597e-02